
# 02 · Baselines — buy_and_hold + equal_weight + random

**Vai trò notebook**: 3 baseline đơn giản nhất — định nghĩa floor (random) + reasonable benchmarks (buy_and_hold, equal_weight).

**Owner**: Person 1 (vừa ramp-up codebase qua baselines)
**Deadline**: 2026-05-22
**Slide chapter**: 4 — Implementation (Strategies → Baselines subsection)

## Mục tiêu
- Cho mỗi agent: one-liner + code reference + sample decision + final result number.
- Cuối notebook so sánh 3 baselines: ai thắng, sao thắng.

## Defense Q&A
- Q: Tại sao bao gồm random agent?
- Q: equal_weight có rebalance không? Tần suất?
- Q: Tại sao buy_and_hold lại thắng nhiều RL/LLM agents?


## Setup


In [ ]:
import sys
from pathlib import Path

_NB_DIR = Path().resolve()
if _NB_DIR.name != "notebooks":
    _NB_DIR = _NB_DIR / "notebooks"
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from _shared import (  # noqa: E402
    AGENT_COLORS,
    BASELINES,
    DATA,
    FIGURES,
    LLM_AGENTS,
    RESULTS,
    RL_AGENTS,
    ROLE_COLORS,
    TRANSCRIPTS,
    assert_frozen_snapshot,
    list_transcript_dates,
    load_curve,
    load_holdings,
    load_metrics_json,
    load_metrics_table,
    load_transcript,
    save_fig,
    setup_matplotlib,
)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

setup_matplotlib()
assert_frozen_snapshot()
metrics = load_metrics_table()
print("snapshot OK · agents:", list(metrics.index))



## TODO-01: buy-and-hold-walkthrough — kiến trúc + sample
- **OWNER**: Person 1   **DEPENDS**: none
- **READ**: `src/baselines.py:BuyAndHold`
- **WRITE**:
  - Markdown: one-liner ("Mua đều 5 ticker ngày đầu test, giữ nguyên đến hết")
  - Code cell: load `BuyAndHold` class, in ra docstring + `.decide()` source
  - Code cell: run `BuyAndHold().decide(state_t0)` cho t=0 → in target weights
- **CONSTRAINTS**:
  - Target weights at t=0 phải uniform 1/5 = 0.2 cho mỗi ticker
  - At t>0 không rebalance → weights drift theo price action
- **VALIDATE**:
  ```python
  bh = BuyAndHold()
  w = bh.decide(state_t0)
  assert abs(sum(w.values()) - 1.0) < 1e-6
  assert all(abs(v - 0.2) < 1e-6 for v in w.values())
  ```
- **PATTERN**: `src/baselines.py` đã có class sẵn — chỉ cần load + show
- **DEFENSE Q&A**: "Buy_and_hold rebalance không?" → KHÔNG; mua đều t=0, drift theo giá đến hết.


In [ ]:
# TODO-01: BuyAndHold walkthrough
from src.baselines import BuyAndHold
import inspect
# print docstring + source
print(inspect.getsource(BuyAndHold))



## TODO-02: equal-weight-walkthrough
- **OWNER**: Person 1   **DEPENDS**: TODO-01
- **READ**: `src/baselines.py:EqualWeightRebalance`
- **WRITE**: tương tự TODO-01 cho EqualWeight
- **CONSTRAINTS**:
  - Khác buy_and_hold: rebalance về 1/N **mỗi ngày** (constant weight)
  - Mỗi rebalance phát sinh fee → total_cost cao hơn buy_and_hold ~10x
- **VALIDATE**:
  ```python
  ew_cost = metrics.loc['equal_weight', 'total_cost']
  bh_cost = metrics.loc['buy_and_hold', 'total_cost']
  assert ew_cost > bh_cost  # rebalance phát fee
  ```
- **DEFENSE Q&A**: "Equal_weight vs buy_and_hold khác gì?" → rebalance vs buy-and-forget.


In [ ]:
# TODO-02: EqualWeight walkthrough
from src.baselines import EqualWeightRebalance
# ...



## TODO-03: random-walkthrough
- **OWNER**: Person 1   **DEPENDS**: TODO-01
- **READ**: `src/baselines.py:RandomAgent`
- **WRITE**: walkthrough + show seed → reproducibility
- **CONSTRAINTS**:
  - Random agent dùng `env.np_random` (seeded) → same seed = same trajectory
  - Show 3 sample decisions: random nhưng deterministic given seed
  - Note: random agent return -10.32% — định nghĩa floor; mọi agent thực phải beat random
- **VALIDATE**:
  ```python
  assert metrics.loc['random', 'cumulative_return'] < 0  # random ≈ lose money on average
  ```
- **DEFENSE Q&A**: "Tại sao có random agent?" → định nghĩa null hypothesis — bất kỳ phương pháp nào thắng random mới có giá trị.


In [ ]:
# TODO-03: RandomAgent walkthrough
from src.baselines import RandomAgent
# ...



## TODO-04: baselines-comparison-figure
- **OWNER**: Person 1   **DEPENDS**: TODO-01..03
- **WRITE**: `report/figures/02__baselines_compare.png`
- **CONSTRAINTS**:
  - 2-panel: (a) equity curves overlaid (3 baselines), (b) bar chart cum_return + Sharpe
  - Color = AGENT_COLORS
  - Title VI: "So sánh 3 baselines — buy_and_hold, equal_weight, random"
- **VALIDATE**: file exists, ≥ 80KB
- **DEFENSE Q&A**: "Trực quan baselines?" → mở figure này


In [ ]:
# TODO-04: baselines comparison figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# ...



## TODO-05: baselines-summary-table
- **OWNER**: Person 1   **DEPENDS**: TODO-04
- **WRITE**: `report/figures/02__baselines_summary.md`
- **CONSTRAINTS**:
  - Bảng 4 cols × 3 rows: agent, cum_return, sharpe, total_cost
  - Đọc từ `metrics`, KHÔNG hardcode
  - Format cum_return %, total_cost VND
- **VALIDATE**: file ≥ 200 chars, 3 baseline rows
- **DEFENSE Q&A**: "Số cụ thể của baselines?" → table này


In [ ]:
# TODO-05: baselines summary table → markdown file
pass



## Defense Q&A — câu trả lời sẵn

> **Q1: Tại sao có random agent?**
> A: Định nghĩa floor — null hypothesis. Bất kỳ phương pháp nào (RL/LLM) phải beat random về cum_return và Sharpe; nếu không tức là không học được gì.
> Evidence: random −10.32% Sharpe −0.45 (table TODO-05).

> **Q2: equal_weight rebalance tần suất?**
> A: Daily. Mỗi phiên kiểm tra weights, nếu drift khỏi 1/N do price action thì trade về 1/N. Phát sinh fee mỗi ngày → total_cost ~10x buy_and_hold.
> Evidence: `src/baselines.py:EqualWeightRebalance` + TODO-02 cost comparison.

> **Q3: Buy_and_hold thắng nhiều RL/LLM — explain?**
> A: Bull market 2025-2026 (VN30 +103%), passive index nắm đủ beta. Active strategies (RL/LLM) tốn fee + bị whipsaw khi market trend mạnh.
> Evidence: TODO-04 equity curve — buy_and_hold leo dần monotonic trong test window.
